# N=100 three-cap long Adam GPU continuation

## Mandatory contract for users and AI agents

**This notebook follows `scripts/templates/boilerplate_run.ipynb`, the source of truth for run notebooks.** Keep its cell order, headings, guards, and adapter calls. Do not add, remove, reorder, merge, or redesign sections without the user's explicit permission in the current conversation.

The completed/running `u40` result is retained, while the revised `u1280` and `u160` configurations run in that order on `bar`'s explicitly selected GPU. Each cap uses 50 new Fourier starts plus five smaller perturbations of each of the top five stored controls with the same cap, smoothness, and sharpness (25 queried starts, 75 total). Stored controls cold-start with fresh Adam state. At learning-rate boundaries, stable runs are saved and removed from the next device batch; each revised cap has a five-hour batch limit.

In [ ]:
from ofc.notebook_workflow import RunNotebook

run_name = "N100_three_cap_adam_revised_schedule_gpu"
workflow_u40 = RunNotebook(f"{run_name}_u40")
workflow_u160 = RunNotebook(f"{run_name}_u160")
workflow_u1280 = RunNotebook(f"{run_name}_u1280")
workflow = workflow_u40

## Create the immutable configs

Edit the exposed mappings, then activate once. All three configs explicitly request `device: gpu`. Every stored-control query explicitly uses `resume_optimizer: false`, so Adam starts with count zero and zero moments.

In [ ]:
Activated = False

description = "N=100 long Adam continuation of three selected caps with boundary pruning and tolerance auto-halt."
reuse_existing = False
common_parameters = {
    "N": 100,
    "t_interval": 4.0,
    "r_bg": -0.008716,
    "u_isbound": True,
    "v_isbound": True,
    "v_max": 1000.0,
    "slew_limit": 0.05,
    "optimizer": "adam",
    "schedule": [(1_000, 1.0), (50_000, 0.1), (100_000, 0.1), (1_000_000, 0.1)],
    "adam_eps": 1e-8,
    "u_smooth": None,
    "v_smooth": None,
    "u_sharp": None,
    "v_sharp": None,
    "block_size": 1_000,
    "J_tol": 1e-5,
    "u_tol": 1e-4,
    "v_tol": 1e-4,
    "projected_gradient_tol": 1e-4,
    "projected_gradient_alpha": 1.0,
}
parameters_u40 = {
    **common_parameters,
    "u_max": 40.0,
    "adam_learning_rate": 0.1,
    "adam_beta1": 0.9,
    "adam_beta2": 0.99,
    "smoothness": 1.25e-7,
    "sharpness": 5e-8,
}
parameters_u160 = {
    **common_parameters,
    "schedule": [(1_000, 1.0), (30_000, 0.1), (60_000, 0.5)],
    "u_max": 160.0,
    "adam_learning_rate": 0.02,
    "adam_beta1": 0.95,
    "adam_beta2": 0.99,
    "smoothness": 1.25e-7,
    "sharpness": 1.25e-8,
}
parameters_u1280 = {
    **common_parameters,
    "schedule": [(1_000, 1.0), (30_000, 0.1), (60_000, 0.5)],
    "u_max": 1280.0,
    "adam_learning_rate": 0.1,
    "adam_beta1": 0.95,
    "adam_beta2": 0.999,
    "smoothness": 2.5e-7,
    "sharpness": 2.5e-8,
}
runtime = {
    "initialisations": 50,
    "fourier_num_modes": 5,
    "fourier_rms_amplitude": 0.3,
    "fourier_intensity_fraction": 0.3,
    "use_jit": True,
    "use_x64": True,
    "device": "gpu",
    "concurrent_workers": 1,
    "max_cases_per_batch": None,
    "auto_halt": True,
    "database": "results/results.sqlite3",
}
runtime_u160 = {**runtime, "max_batch_elapsed_seconds": 5 * 60 * 60}
runtime_u1280 = {**runtime, "max_batch_elapsed_seconds": 5 * 60 * 60}
perturbation_levels = [0.0005, 0.001, 0.0025, 0.005, 0.01]
query_u40 = {
    "where": {"status": "complete", "N": 100, "u_max": 40.0, "smoothness": 1.25e-7, "sharpness": 5e-8},
    "limit": 5,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "resume_optimizer": False,
    "perturbed": True,
    "perturbation_levels": perturbation_levels,
}
query_u160 = {
    "where": {"status": "complete", "N": 100, "u_max": 160.0, "smoothness": 1.25e-7, "sharpness": 1.25e-8},
    "limit": 5,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "resume_optimizer": False,
    "perturbed": True,
    "perturbation_levels": perturbation_levels,
}
query_u1280 = {
    "where": {"status": "complete", "N": 100, "u_max": 1280.0, "smoothness": 2.5e-7, "sharpness": 2.5e-8},
    "limit": 5,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "resume_optimizer": False,
    "perturbed": True,
    "perturbation_levels": perturbation_levels,
}
config_document_u40 = workflow_u40.create_config(
    activated=Activated, description=description, parameters=parameters_u40, runtime=runtime,
    initialization_query=query_u40, reuse_existing=reuse_existing,
)
config_document_u160 = workflow_u160.create_config(
    activated=Activated, description=description, parameters=parameters_u160, runtime=runtime_u160,
    initialization_query=query_u160, reuse_existing=reuse_existing,
)
config_document_u1280 = workflow_u1280.create_config(
    activated=Activated, description=description, parameters=parameters_u1280, runtime=runtime_u1280,
    initialization_query=query_u1280, reuse_existing=reuse_existing,
)

## Run directly on `bar`'s GPU (detached)

This verifies JAX CUDA visibility, then runs the revised `u1280` and `u160` configs sequentially in one detached process and one log. It survives notebook, browser, or laptop disconnection.

In [ ]:
Activated = False

queue_id = None
python_executable = None
extra_arguments = []
detached = True
log_path = None

active_queue_id = workflow_u1280.run_on_bar_gpu_group(
    activated=Activated,
    additional_workflows=(workflow_u160,),
    queue_id=queue_id,
    python_executable=python_executable,
    extra_arguments=extra_arguments,
    detached=detached,
    log_path=log_path,
)

## Submit through Slurm (alternative)

In [ ]:
Activated = False


partition = "zen5,epyc"
time = "4-03:00:00"
cpus = 4
memory = "16G"
array = False
array_max_concurrent = None
extra_arguments = []

slurm_queue_u40 = workflow_u40.submit_slurm(activated=Activated, partition=partition, time=time, cpus=cpus, memory=memory, array=array, array_max_concurrent=array_max_concurrent, extra_arguments=extra_arguments)
slurm_queue_u160 = workflow_u160.submit_slurm(activated=Activated, partition=partition, time=time, cpus=cpus, memory=memory, array=array, array_max_concurrent=array_max_concurrent, extra_arguments=extra_arguments)
slurm_queue_u1280 = workflow_u1280.submit_slurm(activated=Activated, partition=partition, time=time, cpus=cpus, memory=memory, array=array, array_max_concurrent=array_max_concurrent, extra_arguments=extra_arguments)

## Query persisted data

This combines the three immutable configs from one shared queue ID so every plot below is produced inside this notebook. In a fresh kernel, set `queue_id` explicitly or leave it `None` to select the latest common execution rank.

In [ ]:
queue_id = None
config_run_rank = 1
statuses = None
filters = {'u_max':1280}
sweep_parameters = ["u_max"]
require_saved_stage = True
allow_missing = True  # Show completed/saved caps while later sequential configs have not started.
limit = None
order_by = "run_id"
descending = False

query_result = workflow_u40.query_group(
    additional_workflows=(workflow_u160, workflow_u1280),
    queue_id=queue_id,
    config_run_rank=config_run_rank,
    statuses=statuses,
    filters=filters,
    sweep_parameters=sweep_parameters,
    require_saved_stage=require_saved_stage,
    allow_missing=allow_missing,
    limit=limit,
    order_by=order_by,
    descending=descending,
)

## Figure display and saving

In [ ]:
save_figure = None
figure_format = "png"
preview_dpi = 240
save_dpi = 600

## Figure 1 — convergence

In [ ]:
sweep_parameter = "u_max"
figure_1 = query_result.plot_convergence(
    sweep_parameter=sweep_parameter, log_base_x=None, log_base_y=None,
    base_x="axis", base_y="axis", x_multiplier=1, y_multiplier=1,
    x_range=None, y_range=None, x_label=None, y_label=None,
)
workflow.present_figure(figure_1, "01_convergence", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Figure 2 — objective strip plot and seed sensitivity

In [ ]:
sweep_parameter = "u_max"
figure_2 = query_result.plot_distribution(
    sweep_parameter=sweep_parameter, log_base_y=None, base_y="axis",
    y_multiplier=1, y_range=None, x_label=None, y_label=None, point_size=24,
    line_alpha=0.22, seed_sensitivity_log_base_y=10,
    seed_sensitivity_base_y=None, seed_sensitivity_y_multiplier=1,
    seed_sensitivity_y_range=None, seed_sensitivity_tolerance=0.01,
)
workflow.present_figure(figure_2, "02_distribution", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Figure 3 — best controls

In [ ]:
sweep_parameter = "u_max"
figure_3 = query_result.plot_controls(
    sweep_parameter=sweep_parameter, log_base_x=None, log_base_y=None,
    base_x="axis", base_y="axis", x_multiplier=1, y_multiplier=1,
    x_range=None, y_range=None, x_label=None, y_label=None,
)
workflow.present_figure(figure_3, "03_controls", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Single sweep summary

In [ ]:
single_sweep_parameter = "u_max"
history_points = 1200
single_sweep_figure = query_result.plot_single_sweep_summary(sweep_parameter=single_sweep_parameter, history_points=history_points)
workflow.present_figure(single_sweep_figure, "04_single_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Double sweep summary

In [ ]:
separate_sweep_parameter = "u_max"
colour_sweep_parameter = "adam_learning_rate"
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(separate_sweep_parameter=separate_sweep_parameter, colour_sweep_parameter=colour_sweep_parameter, history_points=history_points)
workflow.present_figure(double_sweep_figure, "05_double_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "u_max"
column_sweep_parameter = "smoothness"
colour_sweep_parameter = "adam_learning_rate"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(row_sweep_parameter=row_sweep_parameter, column_sweep_parameter=column_sweep_parameter, colour_sweep_parameter=colour_sweep_parameter, history_points=history_points)
workflow.present_figure(triple_sweep_figure, "06_triple_sweep_summary", save_figure=save_figure, figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi)